# 行列へのノイズ注入（injection_error関数）の挙動デモ

このノートブックでは、PyTorchの行列にノイズを加える `injection_error` 関数の挙動を具体的に可視化・比較します。

- サンプル行列の作成
- ノイズ注入関数の定義
- ノイズ注入前後の行列の可視化
- ノイズ強度（sigma）を変化させた場合の比較

In [1]:
# 必要なライブラリのインポート
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

In [12]:
# サンプル行列の作成
# ランダムな行列（正負混在、0含む）
torch.manual_seed(42)
A = torch.randn(8, 8)
A[::2, ::1] = 0  # 一部を0に
print("元の行列A:")
print(A)

元の行列A:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347, -0.4879],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-1.4032,  0.0360, -0.0635,  0.6756, -0.0978,  1.8446, -1.1845,  1.3835]])


In [13]:
# injection_error関数の定義
def injection_error(
    params: torch.Tensor,
    sigma: float,
    symmetric: bool = True,
    eps: float = 1e-12,
):
    if sigma is None or sigma <= 0.0:
        return params
    # 1. 正規化スケール決定
    if symmetric:
        scale = params.abs().max().clamp(min=eps)
        w_norm = params / scale
    else:
        scale = params.max().clamp(min=eps)
        w_norm = params / scale
    # 2. 0重みマスク
    mask = (w_norm != 0).float()
    # 3. 正規化空間で誤差注入
    noise = torch.randn_like(w_norm) * sigma
    w_norm_noisy = w_norm + noise * mask
    # 4. クリップ
    if symmetric:
        w_norm_noisy = torch.clamp(w_norm_noisy, -1.0, 1.0)
    else:
        w_norm_noisy = torch.clamp(w_norm_noisy, 0.0, 1.0)
    # 5. 元スケールへ復元
    params_noisy = w_norm_noisy * scale
    return params_noisy

In [14]:
# 行列にノイズを加える処理の実行
sigma = 0.05
A_noisy = injection_error(A, sigma=sigma, symmetric=True)
print("ノイズ注入後の行列A_noisy:")
print(A_noisy)

ノイズ注入後の行列A_noisy:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.7704,  1.7043, -0.3865, -1.3584, -0.6669, -0.6275, -1.0030,  0.6828],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.3581,  1.2453,  0.4991,  1.3988, -0.4144, -0.0461, -0.1099,  0.9112],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-1.5772,  1.0755, -0.8368, -0.5822, -1.2460,  2.1228, -1.2348, -0.5201],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-1.4046,  0.0616, -0.0494,  0.7567,  0.0184,  1.8807, -1.1081,  1.4272]])
